# Tonality Features (IEMOCAP)

This notebook extracts chroma and tonnetz summaries.
Each utterance becomes one training row for downstream SER models.

In [1]:
from pathlib import Path

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "tonality"
OUT_FILE = "tonality_features.csv"

# Audio + feature params
TARGET_SR = 16_000
HOP_LENGTH = 512
N_FFT = 2048
STATS = ("mean", "std", "p10", "p50", "p90")

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def summarize_curve(prefix: str, values: np.ndarray, stats: tuple[str, ...]) -> dict[str, float]:
    # Summary stats ignoring NaNs
    vec = np.asarray(values, dtype=float).ravel()
    vec = vec[~np.isnan(vec)]
    if vec.size == 0:
        return {f"{prefix}_{stat}": float("nan") for stat in stats}

    summary: dict[str, float] = {}
    for stat in stats:
        key = f"{prefix}_{stat}"
        if stat == "mean":
            summary[key] = float(np.mean(vec))
        elif stat == "std":
            summary[key] = float(np.std(vec))
        elif stat.startswith("p") and stat[1:].isdigit():
            percentile = int(stat[1:])
            summary[key] = float(np.percentile(vec, percentile))
        else:
            raise ValueError(f"Unknown stat: {stat}")
    return summary


def extract_tonality_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=sr,
        hop_length=HOP_LENGTH,
        n_fft=N_FFT,
    )
    harmonic = librosa.effects.harmonic(audio)
    tonnetz = librosa.feature.tonnetz(y=harmonic, sr=sr)

    features: dict[str, float] = {}
    for idx, curve in enumerate(chroma):
        features.update(summarize_curve(f"chroma_{idx:02d}", curve, STATS))
    for idx, curve in enumerate(tonnetz):
        features.update(summarize_curve(f"tonnetz_{idx:02d}", curve, STATS))
    return features


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: valid emotion + agreement > 0
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


In [5]:
rows: list[dict[str, float | str | int]] = []
missing: list[str] = []

for _, row in df.iterrows():
    rel_path = row["path"]
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        missing.append(str(audio_path))
        continue

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_tonality_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": str(rel_path),
        "session": int(row["session"]),
        "method": row["method"],
        "gender": row["gender"],
        "emotion": row["emotion"],
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    rows.append(record)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape
